# 01 · 分箱模块（Binning）功能演示

演示 hscredit 18 种分箱算法的统一入口 OptimalBinning、二维交互分箱、单调性约束、双 API 风格、三种 transform 输出与规则导出。

In [1]:
import warnings, os
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import hscredit

# 路径约定：从 notebooks/ 目录运行，数据在 ../examples，产物输出到 model_report/
DATA = os.path.join("..", "examples", "hscredit_yyp.xlsx")
if not os.path.exists(DATA):
    DATA = os.path.join("examples", "hscredit_yyp.xlsx")
OUT = "model_report"
os.makedirs(OUT, exist_ok=True)

df = pd.read_excel(DATA)
df["放款时间"] = pd.to_datetime(df["放款时间"])
y = df["FPD"].astype(int)
NUM_FEATURES = ["珊瑚92", "青云24", "衡枢鉴真分老客版", "占信V3", "天创小额网贷分", "近六个月非银多头机构数"]
CAT_FEATURE = "商品类别"
print("数据形状:", df.shape)
print("坏样本率: {:.4f}".format(y.mean()))
df.head()

数据形状: (970, 18)
坏样本率: 0.1402


,客户编号,放款时间,放款金额,商品类别,MOB1,CURRENT_DPD,中智小牛分C3,珊瑚92,极光欺诈分6v1,青云24,占信V3,轻花老客海纳子分V1,天创小额网贷分,近六个月非银多头机构数,手机号近一个月非银多头机构数,身份证近一个月非银多头机构数,衡枢鉴真分老客版,FPD
0,1985945640026276096,2026-02-03,1399,礼包,0,0,NaN,NaN,NaN,656,NaN,NaN,630,51,15,15,0.0242,0
1,1985972188268592896,2026-02-04,1399,礼包,0,0,NaN,NaN,NaN,565,NaN,NaN,583,56,6,18,0.0492,0
2,1986034700861140992,2025-11-06,3960,珠宝首饰,0,0,NaN,NaN,NaN,708,NaN,NaN,764,68,17,20,0.0546,0
3,1986264852923760896,2025-11-06,3960,珠宝首饰,0,0,NaN,NaN,NaN,555,NaN,NaN,712,45,15,15,0.0899,0
4,1986265696509906944,2026-01-26,1399,礼包,0,0,NaN,NaN,NaN,581,NaN,NaN,641,67,32,32,0.0678,0


## 1. 18 种分箱方法统一入口 OptimalBinning
通过 `method` 参数选择分箱算法，所有方法共享统一的 fit/transform/get_bin_table 接口。

In [2]:
from hscredit.core.binning import OptimalBinning

X = df[NUM_FEATURES]
methods = ['uniform','quantile','tree','chi','best_ks','best_iv','mdlp','or_tools','cp_sat',
           'cart','kmeans','monotonic','genetic','smooth','kernel_density','best_lift','target_bad_rate']
summary = []
for m in methods:
    b = OptimalBinning(method=m, max_n_bins=5)
    b.fit(X, y)
    bt = b.get_bin_table('衡枢鉴真分老客版')
    summary.append({'分箱方法': m, '分箱数': int((bt['分箱'] >= 0).sum()), '指标IV值': round(float(bt['指标IV值'].iloc[0]), 4)})
iv_compare = pd.DataFrame(summary).sort_values('指标IV值', ascending=False).reset_index(drop=True)
iv_compare

,分箱方法,分箱数,指标IV值
0,tree,5,0.2770
1,best_iv,5,0.2714
2,kernel_density,4,0.2693
3,cart,5,0.2626
4,mdlp,5,0.2622
5,monotonic,5,0.2616
6,genetic,5,0.2569
7,best_ks,5,0.2536
8,smooth,5,0.2527
9,kmeans,5,0.2467


## 2. 单特征分箱表（中文列名 + compute_bin_stats 指标）

In [3]:
binner = OptimalBinning(method='best_iv', max_n_bins=5)
binner.fit(X, y)
bin_table = binner.get_bin_table('衡枢鉴真分老客版')
bin_table

,分箱,分箱标签,样本总数,好样本数,坏样本数,样本占比,好样本占比,坏样本占比,坏样本率,分档WOE值,分档IV值,指标IV值,LIFT值,坏账改善,累积LIFT值,累积坏账改善,累积好样本数,累积坏样本数,分档KS值
0,0,"[-inf, 0.0694)",358,327,31,0.3691,0.3921,0.2279,0.0866,-0.5424,0.0890,0.2714,0.6176,-0.2237,0.6176,-0.2237,327,31,0.1641
1,1,"[0.0694, 0.1157)",321,278,43,0.3309,0.3333,0.3162,0.1340,-0.0528,0.0009,0.2714,0.9554,-0.0220,0.7773,-0.5196,605,74,0.1813
2,2,"[0.1157, 0.1566)",165,141,24,0.1701,0.1691,0.1765,0.1455,0.0429,0.0003,0.2714,1.0374,0.0077,0.8282,-1.1510,746,98,0.1739
3,3,"[0.1566, 0.1931)",77,57,20,0.0794,0.0683,0.1471,0.2597,0.7663,0.0603,0.2714,1.8526,0.0735,0.9138,-1.6200,803,118,0.0952
4,4,"[0.1931, +inf)",49,31,18,0.0505,0.0372,0.1324,0.3673,1.2700,0.1209,0.2714,2.6200,0.0862,1.0000,1.0000,834,136,0.0000


## 3. 三种 transform 输出：indices / bins / woe

In [4]:
idx = binner.transform(X, metric='indices')
lab = binner.transform(X, metric='bins')
woe = binner.transform(X, metric='woe')
print('indices:'); display(idx.head(3))
print('bins(分箱标签，逐样本):'); display(lab.head(3))
print('woe:'); display(woe.head(3).round(4))

indices:


,珊瑚92,青云24,衡枢鉴真分老客版,占信V3,天创小额网贷分,近六个月非银多头机构数
0,-1,1,0,-1,0,0
1,-1,1,0,-1,0,0
2,-1,1,0,-1,1,0


bins(分箱标签，逐样本):


,珊瑚92,青云24,衡枢鉴真分老客版,占信V3,天创小额网贷分,近六个月非银多头机构数
0,missing,"[484.07, +inf)","[-inf, 0.0694)",missing,"[-inf, 686)","[-inf, 71.5)"
1,missing,"[484.07, +inf)","[-inf, 0.0694)",missing,"[-inf, 686)","[-inf, 71.5)"
2,missing,"[484.07, +inf)","[-inf, 0.0694)",missing,"[686, +inf)","[-inf, 71.5)"


woe:


,珊瑚92,青云24,衡枢鉴真分老客版,占信V3,天创小额网贷分,近六个月非银多头机构数
0,0.0911,0.0280,-0.5424,-22.8217,0.2239,-0.1426
1,0.0911,0.0280,-0.5424,-22.8217,0.2239,-0.1426
2,0.0911,0.0280,-0.5424,-22.8217,-0.1137,-0.1426


## 4. 双 API 风格：sklearn 风格 vs scorecardpipeline 风格

In [5]:
# sklearn 风格
b_sklearn = OptimalBinning(method='best_iv').fit(X, y)
# scorecardpipeline 风格（目标列在 DataFrame 中）
df_with_target = X.copy(); df_with_target['target'] = y.values
b_scp = OptimalBinning(method='best_iv', target='target').fit(df_with_target)
print('两种风格 IV 一致:',
      np.isclose(b_sklearn.get_bin_table('青云24')['指标IV值'].iloc[0],
                 b_scp.get_bin_table('青云24')['指标IV值'].iloc[0]))

两种风格 IV 一致: True


## 5. 单调性约束分箱（ascending / descending / peak / valley / auto）

In [6]:
mono_tables = {}
for mono in ['ascending','descending','auto']:
    bm = OptimalBinning(method='best_iv', monotonic=mono, max_n_bins=6).fit(X, y)
    bt = bm.get_bin_table('衡枢鉴真分老客版')
    valid = bt[bt['分箱'] >= 0]
    mono_tables[mono] = valid[['分箱标签','样本总数','坏样本率','分档WOE值']].reset_index(drop=True)
mono_tables['descending']

,分箱标签,样本总数,坏样本率,分档WOE值
0,"[-inf, 0.0759)",419,0.1074,-0.3040
1,"[0.0759, +inf)",551,0.1652,0.1932


## 6. 类别型特征分箱

In [7]:
Xc = df[NUM_FEATURES + [CAT_FEATURE]]
bc = OptimalBinning(method='best_iv', max_n_bins=5).fit(Xc, y)
bc.get_bin_table(CAT_FEATURE)

,分箱,分箱标签,样本总数,好样本数,坏样本数,样本占比,好样本占比,坏样本占比,坏样本率,分档WOE值,分档IV值,指标IV值,LIFT值,坏账改善,累积LIFT值,累积坏账改善,累积好样本数,累积坏样本数,分档KS值
0,0,"家用电器,智能设备,电脑数码",26,24,2,0.0268,0.0288,0.0147,0.0769,-0.6713,0.0094,0.0380,0.5486,-0.0124,0.5486,-0.0124,24,2,0.0141
1,1,手机通讯,141,126,15,0.1454,0.1511,0.1103,0.1064,-0.3147,0.0128,0.0380,0.7588,-0.0410,0.7260,-0.0570,150,17,0.0549
2,2,礼包,189,166,23,0.1948,0.1990,0.1691,0.1217,-0.1629,0.0049,0.0380,0.8680,-0.0320,0.8014,-0.1152,316,40,0.0848
3,3,珠宝首饰,614,518,96,0.6330,0.6211,0.7059,0.1564,0.1280,0.0108,0.0380,1.1152,0.1986,1.0000,1.0000,834,136,0.0000


## 7. 自动选择最优分箱方法

In [8]:
best = OptimalBinning.auto_select_method(X, y, '衡枢鉴真分老客版')
print('auto_select_method 选出:', best)

auto_select_method 选出: tree


## 8. 二维交互分箱 OptimalBinning2D

In [9]:
from hscredit.core.binning import OptimalBinning2D
b2d = OptimalBinning2D(max_n_bins=4)
b2d.fit(df[['衡枢鉴真分老客版','青云24']], y)
out2d = b2d.transform(df[['衡枢鉴真分老客版','青云24']], metric='woe')
print('2D woe 输出形状:', out2d.shape)
out2d.head()

2D woe 输出形状: (970, 1)


,衡枢鉴真分老客版X青云24
0,-0.2890
1,-1.0608
2,-0.2890
3,0.0321
4,0.0321


## 9. 规则导出/加载（部署一致性）

In [10]:
rules = binner.export()
b_loaded = OptimalBinning(method='best_iv'); b_loaded.load(rules)
same = binner.transform(X, metric='woe').equals(b_loaded.transform(X, metric='woe'))
print('导出/加载后 WOE 转换一致:', same)

导出/加载后 WOE 转换一致: True


## 10. 分箱结果导出到 Excel（model_report）

In [11]:
from hscredit.excel import dataframe2excel
out_path = f"{OUT}/01_binning_bin_tables.xlsx"
dataframe2excel(bin_table, out_path, sheet_name='衡枢鉴真分老客版', title='best_iv 分箱表')
iv_compare.to_excel(f"{OUT}/01_binning_method_iv_compare.xlsx", index=False)
print('已保存:', out_path)

已保存: model_report/01_binning_bin_tables.xlsx
